# Train A Model From `react.xyz` / `product.xyz`

所有输出默认写入：

```text
tests/react_product_xyz_training/
```

它会做完整训练流程：

1. 读取 `tests/data/react.xyz` 和 `tests/data/product.xyz`。
2. 按 frame index 默认切分为：70% train、10% validation、20% test。
3. 训练 `DiffModule`。
4. 用 validation split 监控 loss 并保存 checkpoint。
5. 训练结束后评估 test denoising loss 和一次 inpainting RMSD。
6. 保存：split 文件、checkpoint、CSV loss 日志、运行配置 `run_config.json`。
7. 后续可以用 `inpaint_from_checkpoint.ipynb` 加载 checkpoint 做 product 生成。

## 0. Imports And Paths

请确认 kernel 是你配置好的对应依赖的环境。

In [ ]:
from __future__ import annotations

import csv
import inspect
import json
import math
import sys
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import numpy as np
import torch
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks import Callback, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "train_react_product_xyz.ipynb").exists():
    NOTEBOOK_DIR = Path("tests/react_product_xyz_training").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
PACKAGE_PARENT = REPO_ROOT.parent
if str(PACKAGE_PARENT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PARENT))

from akmcgc.model import EGNN, LEFTNet
from akmcgc.trainer.task import DiffModule

TEST_DIR = NOTEBOOK_DIR
TEST_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT =", REPO_ROOT)
print("TEST_DIR  =", TEST_DIR)
print("torch     =", torch.__version__)

## 1. Configuration

实时查看训练情况的方法：

1. 运行 `trainer.fit(module)` 的cell 时，Lightning 会显示 tqdm 进度条。
2. `LiveLossPrinter` 会每隔若干 batch 打印一次当前 train loss。
3. `CSVLogger` 会把 Lightning 日志写到 `runs/<RUN_NAME>/lightning_logs/.../metrics.csv`。
4. 训练后 notebook 会读取 `history.train_losses` 打印前后 loss 均值。

In [ ]:
# Input files.
REACT_FILE = REPO_ROOT / "tests" / "data" / "react.xyz"
PRODUCT_FILE = REPO_ROOT / "tests" / "data" / "product.xyz"

# Reproducibility and device.
SEED = 42
ACCELERATOR = "cpu"   # change to "gpu" if available
DEVICES = 1
DTYPE = "float64"

# Dedicated output location for this test.
RUN_NAME = "pdau_test"
OUTPUT_DIR = TEST_DIR / "runs" / RUN_NAME
SPLIT_DIR = TEST_DIR / "splits" / f"seed_{SEED}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Split policy.
TRAIN_RATIO = 0.70      # Training set ratio
VAL_RATIO = 0.10        # Validation set ratio
TEST_RATIO = 0.20       # Test set ratio
SHUFFLE_SPLIT = True    # Whether to shuffle the data before splitting
OVERWRITE_SPLITS = True # Whether to overwrite existing splits

# Dataset / graph construction.
CUTOFF = 6.0         # Distance cutoff for connecting edges in the graph. Set to None for no cutoff (fully connected).
MAX_NEIGH = 100      # Maximum number of neighbors per node after cutoff. Set to None for no limit.
BATCH_SIZE = 100      # Batch size for training and evaluation
NUM_WORKERS = 0      # Number of workers for data loading. Set to 0 for debugging or if issues arise.

# Backbone.
MODEL_TYPE = "egnn"  # "egnn" or "leftnet"
HIDDEN_NF = 64      # Hidden node feature dimension
N_LAYERS = 2         # Number of EGNN/LEFTNet layers (not counting input embedding MLP or output MLP)

# Optimization.
LR = 1e-4                    # Learning rate
MAX_EPOCHS = 10              # Maximum number of epochs to train for
LIMIT_TRAIN_BATCHES = 20   # None means full train split every epoch
LIMIT_VAL_BATCHES = 5     # None means full validation split

# Diffusion.
TIMESTEPS = 1000           # Number of diffusion steps during training
SAMPLING_TIMESTEPS = 100   # Number of diffusion steps during sampling (inpainting evaluation)
POS_ONLY = True     # Whether to only predict positions (True) or also atom types (False)

# Post-training evaluation.
RUN_INPAINT_EVAL = True # Whether to run inpainting evaluation after training
INPAINT_TIMESTEPS = 32  # Number of diffusion steps for inpainting evaluation
EVAL_TEST_BATCHES = 5  # None means full test split

seed_everything(SEED, workers=True)
torch.set_default_dtype(torch.float64)

print("REACT_FILE exists   =", REACT_FILE.exists())
print("PRODUCT_FILE exists =", PRODUCT_FILE.exists())
print("OUTPUT_DIR          =", OUTPUT_DIR)
print("SPLIT_DIR           =", SPLIT_DIR)

## 2. Split `react.xyz` / `product.xyz` Into 70/10/20

切分必须按相同 frame index 同步切分 react 和 product，因为第 `i` 帧 react 对应第 `i` 帧 product。


In [3]:
def read_xyz_frames(path: Path) -> List[List[str]]:
    lines = path.read_text().splitlines(keepends=True)
    frames: List[List[str]] = []
    idx = 0
    while idx < len(lines):
        if not lines[idx].strip():
            idx += 1
            continue
        n_atoms = int(lines[idx].strip())
        frame = lines[idx : idx + n_atoms + 2]
        if len(frame) != n_atoms + 2:
            raise ValueError(f"Incomplete frame in {path} at line {idx}")
        frames.append(frame)
        idx += n_atoms + 2
    return frames


def write_xyz_frames(path: Path, frames: List[List[str]], indices: List[int]) -> None:
    with path.open("w") as f:
        for frame_idx in indices:
            f.writelines(frames[frame_idx])


def split_pair_xyz(
    react_file: Path,
    product_file: Path,
    split_dir: Path,
    seed: int,
    train_ratio: float,
    val_ratio: float,
    overwrite: bool,
) -> Dict[str, Path]:
    split_dir.mkdir(parents=True, exist_ok=True)
    output_paths = {
        "train_react": split_dir / "train_react.xyz",
        "train_product": split_dir / "train_product.xyz",
        "val_react": split_dir / "val_react.xyz",
        "val_product": split_dir / "val_product.xyz",
        "test_react": split_dir / "test_react.xyz",
        "test_product": split_dir / "test_product.xyz",
        "indices": split_dir / "split_indices.json",
    }
    if not overwrite and all(path.exists() for path in output_paths.values()):
        return output_paths

    react_frames = read_xyz_frames(react_file)
    product_frames = read_xyz_frames(product_file)
    if len(react_frames) != len(product_frames):
        raise ValueError(f"Frame mismatch: react={len(react_frames)}, product={len(product_frames)}")

    n_frames = len(react_frames)
    indices = list(range(n_frames))
    if SHUFFLE_SPLIT:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)

    n_train = int(n_frames * train_ratio)
    n_val = int(n_frames * val_ratio)
    train_indices = indices[:n_train]
    val_indices = indices[n_train : n_train + n_val]
    test_indices = indices[n_train + n_val :]

    write_xyz_frames(output_paths["train_react"], react_frames, train_indices)
    write_xyz_frames(output_paths["train_product"], product_frames, train_indices)
    write_xyz_frames(output_paths["val_react"], react_frames, val_indices)
    write_xyz_frames(output_paths["val_product"], product_frames, val_indices)
    write_xyz_frames(output_paths["test_react"], react_frames, test_indices)
    write_xyz_frames(output_paths["test_product"], product_frames, test_indices)

    metadata = {
        "source_react": str(react_file),
        "source_product": str(product_file),
        "seed": seed,
        "shuffle": SHUFFLE_SPLIT,
        "n_frames": n_frames,
        "n_train": len(train_indices),
        "n_val": len(val_indices),
        "n_test": len(test_indices),
        "train_indices": train_indices,
        "val_indices": val_indices,
        "test_indices": test_indices,
    }
    output_paths["indices"].write_text(json.dumps(metadata, indent=2))
    return output_paths


split_paths = split_pair_xyz(
    REACT_FILE,
    PRODUCT_FILE,
    SPLIT_DIR,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    overwrite=OVERWRITE_SPLITS,
)

split_meta = json.loads(split_paths["indices"].read_text())
print(json.dumps({k: split_meta[k] for k in ["n_frames", "n_train", "n_val", "n_test"]}, indent=2))
for key, path in split_paths.items():
    print(f"{key:14s}", path)

{
  "n_frames": 1430,
  "n_train": 1000,
  "n_val": 143,
  "n_test": 287
}
train_react    /Users/wx/Desktop/yyxwjq/tests/react_product_xyz_training/splits/seed_42/train_react.xyz
train_product  /Users/wx/Desktop/yyxwjq/tests/react_product_xyz_training/splits/seed_42/train_product.xyz
val_react      /Users/wx/Desktop/yyxwjq/tests/react_product_xyz_training/splits/seed_42/val_react.xyz
val_product    /Users/wx/Desktop/yyxwjq/tests/react_product_xyz_training/splits/seed_42/val_product.xyz
test_react     /Users/wx/Desktop/yyxwjq/tests/react_product_xyz_training/splits/seed_42/test_react.xyz
test_product   /Users/wx/Desktop/yyxwjq/tests/react_product_xyz_training/splits/seed_42/test_product.xyz
indices        /Users/wx/Desktop/yyxwjq/tests/react_product_xyz_training/splits/seed_42/split_indices.json


## 3. Utility Functions And Callbacks

`LossHistory` 和 `LiveLossPrinter` 用来观察训练过程。

- `LossHistory`：保存 batch loss，训练后可以分析。
- `LiveLossPrinter`：训练时实时打印 loss。

In [4]:
class LossHistory(Callback):
    # 记录训练和验证阶段的 loss，后面会写入 csv 并做前后对比。
    def __init__(self) -> None:
        super().__init__()
        self.train_losses: List[float] = []
        self.val_losses: List[float] = []

    # Lightning 每跑完一个 train batch 会自动调用这个函数。
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx) -> None:
        del trainer, pl_module, batch, batch_idx
        if isinstance(outputs, dict) and "loss" in outputs:
            self.train_losses.append(float(outputs["loss"].detach().cpu()))

    # Lightning 每跑完一个 validation batch 会自动调用这个函数。
    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0) -> None:
        del trainer, pl_module, batch, batch_idx, dataloader_idx
        if isinstance(outputs, dict) and "val-totloss" in outputs:
            self.val_losses.append(float(outputs["val-totloss"]))


class LiveLossPrinter(Callback):
    # 控制 notebook 里实时打印训练 loss 的频率。
    def __init__(self, every_n_train_batches: int = 10) -> None:
        super().__init__()
        self.every_n_train_batches = every_n_train_batches

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx) -> None:
        del pl_module, batch
        if not isinstance(outputs, dict) or "loss" not in outputs:
            return
        # 第一个 batch 一定打印一次，后面每隔 every_n_train_batches 打印一次。
        if batch_idx == 0 or (batch_idx + 1) % self.every_n_train_batches == 0:
            loss = float(outputs["loss"].detach().cpu())
            print(f"[train] epoch={trainer.current_epoch} batch={batch_idx + 1} loss={loss:.6f}")


def tensor_summary(name: str, value: torch.Tensor) -> str:
    # 把 tensor 的名字、shape、dtype、device 压成一行，方便 debug batch 结构。
    return f"{name:14s} shape={tuple(value.shape)!s:16s} dtype={str(value.dtype):14s} device={value.device}"


def mean(values: List[float]) -> float:
    # 空列表时返回 nan，避免 np.mean([]) 带来不直观的结果。
    return float(np.mean(values)) if values else math.nan


def limited_batches(loader: Iterable[Dict], n_batches: Optional[int]) -> Iterable[Dict]:
    # 只取 dataloader 前 n_batches 个 batch；None 表示全取。
    for idx, batch in enumerate(loader):
        if n_batches is not None and idx >= n_batches:
            break
        yield batch

## 4. Build `DiffModule`

这里用 split 后的 train/val/test 文件构建训练模块。 这里的多数参数都在cell 2中设置过。

In [5]:
def build_diff_module() -> DiffModule:
    training_config: Dict[str, Any] = {
        "train_react_file": str(split_paths["train_react"]),
        "train_product_file": str(split_paths["train_product"]),
        "val_react_file": str(split_paths["val_react"]),
        "val_product_file": str(split_paths["val_product"]),
        "test_react_file": str(split_paths["test_react"]),
        "test_product_file": str(split_paths["test_product"]),
        "cutoff": CUTOFF,
        "max_neigh": MAX_NEIGH,
        "r_fixed": True,
        "r_pbc": True,
        "device": "cpu",
        "dtype": DTYPE,
        "num_elements": 118,
        "bz": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "clip_grad": False,
        "lr_schedule_type": None,
        "sampling_timesteps": SAMPLING_TIMESTEPS,
    }
    node_nf = 3 + training_config["num_elements"] + 1
    in_hidden_with_time = HIDDEN_NF + 1

    if MODEL_TYPE == "egnn":
        model = EGNN
        model_config = {
            "in_node_nf": in_hidden_with_time,
            "in_edge_nf": 0,
            "hidden_nf": HIDDEN_NF,
            "edge_hidden_nf": max(16, HIDDEN_NF // 2),
            "act_fn": "swish",
            "n_layers": N_LAYERS,
            "attention": False,
            "out_node_nf": None,
            "tanh": True,
            "coords_range": 5.0,
            "norm_constant": 1.0,
            "inv_sublayers": 1,
            "sin_embedding": False,
            "normalization_factor": 1.0,
            "aggregation_method": "mean",
            "reflect_equiv": True,
        }
    elif MODEL_TYPE == "leftnet":
        model = LEFTNet
        model_config = {
            "pos_require_grad": False,
            "cutoff": CUTOFF,
            "num_layers": N_LAYERS,
            "hidden_channels": HIDDEN_NF,
            "num_radial": 32,
            "in_hidden_channels": in_hidden_with_time,
            "reflect_equiv": True,
            "legacy": True,
            "update": True,
            "pos_grad": False,
            "single_layer_output": True,
            "object_aware": True,
        }
    else:
        raise ValueError(f"Unknown MODEL_TYPE: {MODEL_TYPE}")

    return DiffModule(
        model_config=model_config,
        optimizer_config={"lr": LR, "betas": [0.9, 0.999], "weight_decay": 0.0},
        training_config=training_config,
        node_nfs=[node_nf],
        edge_nf=0,
        condition_nf=0,
        fragment_names=["IS", "FS"],
        pos_dim=3,
        update_pocket_coords=True,
        condition_time=True,
        edge_cutoff=None,
        norm_values=(1.0, 1.0, 1.0),
        norm_biases=(0.0, 0.0, 0.0),
        noise_schedule="cosine",
        timesteps=TIMESTEPS,
        precision=1e-5,
        loss_type="l2",
        pos_only=POS_ONLY,
        model=model,
        eval_epochs=5,
    )

## 5. Inspect One Batch

确认数据切分后的 DataLoader 能正常返回 batch。

In [6]:
module = build_diff_module()
module.setup("fit")
module.setup("test")
print("train dataset size =", len(module.train_dataset))
print("val dataset size   =", len(module.val_dataset))
print("test dataset size  =", len(module.test_dataset))

train_loader = module.train_dataloader()
val_loader = module.val_dataloader()
test_loader = module.test_dataloader()
fixed_test_batch = next(iter(test_loader))

print("\n[fixed test batch]")
for key in ("h", "pos", "edge_index", "cell_offsets", "cell", "pbc", "fragment", "mask"):
    print(tensor_summary(key, fixed_test_batch[key]))

src, dst = fixed_test_batch["edge_index"]
assert torch.allclose(fixed_test_batch["h"][:, :3], fixed_test_batch["pos"])
assert torch.all(fixed_test_batch["mask"][src] == fixed_test_batch["mask"][dst])
assert torch.all(fixed_test_batch["fragment"][src] == fixed_test_batch["fragment"][dst])
print("Batch contract passed.")

train dataset size = 1000
val dataset size   = 143
test dataset size  = 287

[fixed test batch]
h              shape=(15800, 122)     dtype=torch.float64  device=cpu
pos            shape=(15800, 3)       dtype=torch.float64  device=cpu
edge_index     shape=(2, 445618)      dtype=torch.int64    device=cpu
cell_offsets   shape=(445618, 3)      dtype=torch.float64  device=cpu
cell           shape=(100, 2, 3, 3)   dtype=torch.float64  device=cpu
pbc            shape=(100, 2, 3)      dtype=torch.bool     device=cpu
fragment       shape=(15800,)         dtype=torch.int64    device=cpu
mask           shape=(15800,)         dtype=torch.int64    device=cpu
Batch contract passed.


## 6. Evaluation Helpers

优先看 `denoising_coord_loss`。它代表模型预测噪声的能力。

In [7]:
def denoising_coord_loss(module: DiffModule, loader: Iterable[Dict], n_batches: Optional[int], seed: int) -> float:
    # 评估模型在测试集上的去噪能力，不更新参数。
    was_training = module.training
    module.eval()
    torch.manual_seed(seed)
    losses = []
    with torch.no_grad():
        for batch in limited_batches(loader, n_batches):
            # 直接调用 ddpm.forward，得到当前 batch 的扩散 loss 各组成部分。
            loss_terms = module.ddpm(batch, conditions=None)
            # n_is / n_fs 分别是 reactant 和 product 的原子数。
            num_nodes = (batch["n_is"] + batch["n_fs"]).to(loss_terms["error_t"].dtype)
            # error_t 是坐标噪声预测误差；再除以 3*节点数，得到按每个坐标维度归一化后的误差。
            normalized = loss_terms["error_t"] / (module.ddpm.pos_dim * num_nodes)
            losses.append(float(normalized.mean().detach().cpu()))
    module.train(was_training)
    return mean(losses)


def product_rmsd(batch: Dict[str, torch.Tensor], pred_pos: torch.Tensor) -> float:
    # 只看 fragment == 1，也就是 product / final state 部分的坐标误差。
    product_mask = batch["fragment"] == 1
    diff = pred_pos[product_mask] - batch["pos"][product_mask]
    return float(torch.sqrt(torch.mean(torch.sum(diff * diff, dim=1))).detach().cpu())


def inpaint_product_rmsd(module: DiffModule, batch: Dict[str, torch.Tensor], timesteps: int, seed: int) -> float:
    # 固定 reactant(fragment == 0)，只生成 product(fragment == 1)，再计算 product RMSD。
    was_training = module.training
    module.eval()
    torch.manual_seed(seed)
    with torch.no_grad():
        out = module.ddpm.inpaint(batch=batch, conditions=None, frag_fixed=[0], timesteps=timesteps)
    module.train(was_training)
    return product_rmsd(batch, out["pos"])

## 7. Baseline Before Training

In [8]:
initial_test_loss = denoising_coord_loss(module, test_loader, EVAL_TEST_BATCHES, seed=2025)
print("initial test denoising coord loss =", initial_test_loss)

if RUN_INPAINT_EVAL:
    initial_inpaint_rmsd = inpaint_product_rmsd(module, fixed_test_batch, INPAINT_TIMESTEPS, seed=2026)
    print("initial test inpaint product RMSD =", initial_inpaint_rmsd)
else:
    initial_inpaint_rmsd = math.nan

initial test denoising coord loss = 0.9272345777098749
initial test inpaint product RMSD = 32772.501568651525


## 8. Save Run Config

这个文件用于之后加载 checkpoint。`inpaint_from_checkpoint.ipynb` 会读取它。

这里会把训练时的关键超参数一起保存，避免后面加载 `.ckpt` 时手动重建配置出错。

In [ ]:
run_config = {
    "run_name": RUN_NAME,
    "output_dir": str(OUTPUT_DIR),
    "split_dir": str(SPLIT_DIR),
    "seed": SEED,
    "model_type": MODEL_TYPE,
    "hidden_nf": HIDDEN_NF,
    "n_layers": N_LAYERS,
    "cutoff": CUTOFF,
    "max_neigh": MAX_NEIGH,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "dtype": DTYPE,
    "lr": LR,
    "timesteps": TIMESTEPS,
    "sampling_timesteps": SAMPLING_TIMESTEPS,
    "pos_only": POS_ONLY,
    "condition_nf": 0,
    "primary_generation_mode": "inpaint",
    "primary_task": "conditioned reactant->product generation with frag_fixed=[0]",
    "secondary_generation_mode": "sample",
    "split_paths": {k: str(v) for k, v in split_paths.items()},
}
config_path = OUTPUT_DIR / "run_config.json"
config_path.write_text(json.dumps(run_config, indent=2))
print("saved config:", config_path)

## 9. Train

运行这个 cell 时看实时训练情况：

- tqdm 进度条显示 epoch/batch 进度；
- `LiveLossPrinter` 打印类似 `[train] epoch=0 batch=10 loss=...`；
- CSVLogger 写 `metrics.csv`；
- ModelCheckpoint 保存 `.ckpt` 文件。

In [ ]:
history = LossHistory()
live_printer = LiveLossPrinter(every_n_train_batches=10)
checkpoint_callback = ModelCheckpoint(
    dirpath=str(OUTPUT_DIR / "checkpoints"),
    filename="epoch={epoch:03d}-val={val-totloss:.3f}",
    monitor="val-totloss",
    mode="min",
    save_top_k=2,
    save_last=True,
    auto_insert_metric_name=False,
)
logger = CSVLogger(save_dir=str(OUTPUT_DIR / "lightning_logs"), name="csv")

trainer_kwargs = {
    "max_epochs": MAX_EPOCHS,
    "accelerator": ACCELERATOR,
    "devices": DEVICES,
    "logger": logger,
    "enable_checkpointing": True,
    "enable_progress_bar": True,
    "log_every_n_steps": 1,
    "callbacks": [history, live_printer, checkpoint_callback],
    "num_sanity_val_steps": 0,
}
if LIMIT_TRAIN_BATCHES is not None:
    trainer_kwargs["limit_train_batches"] = LIMIT_TRAIN_BATCHES
if LIMIT_VAL_BATCHES is not None:
    trainer_kwargs["limit_val_batches"] = LIMIT_VAL_BATCHES

accepted = set(inspect.signature(Trainer.__init__).parameters)
trainer_kwargs = {k: v for k, v in trainer_kwargs.items() if k in accepted}

module.train()
trainer = Trainer(**trainer_kwargs)
trainer.fit(module)

print("best checkpoint =", checkpoint_callback.best_model_path)
print("last checkpoint =", checkpoint_callback.last_model_path)
print("csv log dir     =", logger.log_dir)

## 10. Save And Inspect Loss History

In [ ]:
loss_csv = OUTPUT_DIR / "loss_history.csv"
with loss_csv.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["kind", "step", "loss"])
    for idx, value in enumerate(history.train_losses):
        writer.writerow(["train", idx, value])
    for idx, value in enumerate(history.val_losses):
        writer.writerow(["val", idx, value])

print("loss history saved to", loss_csv)
print("num train losses =", len(history.train_losses))
print("num val losses   =", len(history.val_losses))
print("first  train losses =", history.train_losses[:5])
print("last  train losses  =", history.train_losses[-5:])
print("mean first =", mean(history.train_losses[:5]))
print("mean last  =", mean(history.train_losses[-5:]))

## 11. Test Evaluation After Training

In [ ]:
final_test_loss = denoising_coord_loss(module, test_loader, EVAL_TEST_BATCHES, seed=123)
print("initial test denoising coord loss =", initial_test_loss)
print("final test denoising coord loss   =", final_test_loss)
print("test loss improvement             =", initial_test_loss - final_test_loss)

if RUN_INPAINT_EVAL:
    final_inpaint_rmsd = inpaint_product_rmsd(module, fixed_test_batch, INPAINT_TIMESTEPS, seed=321)
    print("initial test inpaint product RMSD =", initial_inpaint_rmsd)
    print("final test inpaint product RMSD   =", final_inpaint_rmsd)
    print("inpaint RMSD improvement          =", initial_inpaint_rmsd - final_inpaint_rmsd)
else:
    final_inpaint_rmsd = math.nan

assert np.isfinite(final_test_loss)
assert history.train_losses
assert all(np.isfinite(history.train_losses))

initial test denoising coord loss = 0.9272345777098749
final test denoising coord loss   = 0.7035041434557671
test loss improvement             = 0.22373043425410788
initial test inpaint product RMSD = 32772.501568651525
final test inpaint product RMSD   = 12957.534229549787
inpaint RMSD improvement          = 19814.96733910174


## 12. How To Use The Saved Model File

训练后会得到 checkpoint，例如：

```text
tests/react_product_xyz_training/runs/pdau_test/checkpoints/last.ckpt
```

这个 `.ckpt` 不是“单独一个 product 文件”，而是训练好的模型参数。当前这套测试资产只保留了最自然的反应预测路径：

```text
固定 reactant(fragment == 0) -> inpaint 生成 product(fragment == 1)
```

训练完成后，直接打开：

```text
tests/react_product_xyz_training/inpaint_from_checkpoint.ipynb
```

它会读取当前 run 的 `run_config.json` 和 `.ckpt`，然后写出：

```text
conditioned_reactant.extxyz
predicted_product.extxyz
target_product.extxyz
inpaint_visualization.png
inpaint_summary.json
```